## Donchian Channel Strategy
A volatility-based breakout indicator.

- H Band = Rolling max of OHLC over N periods
- L Band = Rolling min of OHLC over N periods
- M Band = (H Band + L Band) / 2
- P Band = (Close − L Band) / (H Band − L Band)
- W Band = (H Band − L Band) / N-period moving average

<p style="color:steelblue; font-weight:bold">Please run the last cell (donchain4 function) first</p>

In [ ]:
library(TTR)
library(dplyr)
library(zoo)

In [ ]:
ticker <- "MSTR"
data   <- read.csv(sprintf("../../../data/%s.5MIN.csv", ticker))
data$timestamp <- as.POSIXct(data$timestamp)

In [ ]:
compute_donchian <- function(df, window = 12) {
    n           <- nrow(df)
    dc_hband    <- zoo::rollmax(df$high,  window, fill = NA, align = "right")
    dc_lband    <- -zoo::rollmax(-df$low, window, fill = NA, align = "right")
    dc_mband    <- (dc_hband + dc_lband) / 2
    dc_pband    <- (df$close - dc_lband) / (dc_hband - dc_lband)
    roll_mean   <- zoo::rollmean(df$close, window, fill = NA, align = "right")
    dc_wband    <- (dc_hband - dc_lband) / roll_mean
    data.frame(dc_hband, dc_lband, dc_mband, dc_pband, dc_wband)
}

dc_df <- cbind(data, compute_donchian(data, window = 12))

In [ ]:
as_of_date <- "2025-11-20"
temp_plot  <- dc_df[dc_df$timestamp >= as.POSIXct(as_of_date), ]
idx        <- seq_len(nrow(temp_plot))

par(mfrow = c(2, 1), mar = c(2,4,2,1))
plot(idx, temp_plot$close,   type = "l", main = "Donchian Bands", ylab = "Price")
lines(idx, temp_plot$dc_hband, col = "red")
lines(idx, temp_plot$dc_lband, col = "green")
lines(idx, temp_plot$dc_mband, col = "orange")
legend("topleft", legend=c("Close","H","L","M"),
       col=c("black","red","green","orange"), lty=1, bty="n", cex=0.7)
grid()

plot(idx, temp_plot$dc_pband, type="l", col="blue", main="P & W Band", ylab="")
lines(idx, temp_plot$dc_wband, col="purple")
legend("topleft", legend=c("P Band","W Band"),
       col=c("blue","purple"), lty=1, bty="n", cex=0.7)
grid()
par(mfrow = c(1,1))

## Run Backtest

In [ ]:
df_bt <- donchain4(7, data)

## Plot the last period

In [ ]:
temp <- df_bt[df_bt$timestamp >= as.POSIXct(as_of_date), ]
idx  <- seq_len(nrow(temp))

par(mfrow = c(2,1), mar = c(2,4,2,1))
plot(idx, temp$close, type="l", main="Trade Signals", ylab="Price")
lines(idx, temp$dc_hband, col="red")
lines(idx, temp$dc_lband, col="green")
lines(idx, temp$dc_mband, col="orange")
text(idx, temp$close, labels=temp$Trade, cex=0.45, srt=90)
grid()

plot(idx, temp$dc_pband, type="l", col="blue", main="P & W Band", ylab="")
lines(idx, temp$dc_wband, col="purple")
grid()
par(mfrow=c(1,1))

In [ ]:
plot(df_bt$CumuPnL, type="l",
     main="Cumulative PnL", xlab="Bar", ylab="PnL ($)")
grid()

In [ ]:
exits        <- df_bt$Trade %in% c("LExit","SExit")
total_trades <- sum(exits)
cum_ret      <- cumprod(c(1, diff(df_bt$CumuPnL)/head(df_bt$CumuPnL,-1)+1))
peak         <- max(cum_ret)
peak_idx     <- which(cum_ret==peak)[length(which(cum_ret==peak))]
trough       <- min(cum_ret[peak_idx:length(cum_ret)])
max_dd       <- (trough-peak)/peak

metrics <- data.frame(
    Metric = c("Total PnL","Wins","Losses","Win Ratio","Total Trades","Max Drawdown"),
    Value  = c(formatC(sum(df_bt$PnL),format="f",digits=1,big.mark=","),
               sum(df_bt$PnL>0), sum(df_bt$PnL<0),
               sprintf("%.1f%%", sum(df_bt$PnL>0)/total_trades*100),
               total_trades, sprintf("%.1f%%", max_dd*100))
)
print(metrics)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# donchain4 function — run this cell first
# ──────────────────────────────────────────────────────────────────────────────
donchain4 <- function(window, temp_df) {
    df <- temp_df

    df$dc_hband <- zoo::rollmax(df$high,  window, fill=NA, align="right")
    df$dc_lband <- -zoo::rollmax(-df$low, window, fill=NA, align="right")
    df$dc_mband <- (df$dc_hband + df$dc_lband) / 2
    df$dc_pband <- (df$close - df$dc_lband) / (df$dc_hband - df$dc_lband)
    roll_mean   <- zoo::rollmean(df$close, window, fill=NA, align="right")
    df$dc_wband <- (df$dc_hband - df$dc_lband) / roll_mean

    df$Trade    <- ""
    df$PnL      <- 0.0
    short       <- FALSE; long_pos <- FALSE
    enter_price <- 0; shares_traded <- 0; notional <- 10000
    was_in_short <- FALSE; was_in_long <- FALSE
    exit_signal  <- FALSE
    last_dc_pband <- NA; last_dc_wband <- NA

    for (i in seq_len(nrow(df))) {
        if (i <= window) {
            last_dc_pband <- df$dc_pband[i]
            last_dc_wband <- df$dc_wband[i]
            next
        }
        row <- df[i, ]
        if (!short && !long_pos) {
            if (!was_in_short && !is.na(row$dc_pband) && row$dc_pband <= 0.5 &&
                row$close < row$dc_mband && !is.na(row$dc_wband) && row$dc_wband > 0.8) {
                enter_price <- row$close
                shares_traded <- round(notional / row$close, 0)
                short <- TRUE; df$Trade[i] <- "S"
            } else if (!was_in_long && !is.na(row$dc_pband) && row$dc_pband >= 0.5 &&
                       row$close > row$dc_mband && !is.na(row$dc_wband) && row$dc_wband > 0.8) {
                enter_price <- row$close
                shares_traded <- round(notional / row$close, 0)
                long_pos <- TRUE; df$Trade[i] <- "L"
            }
            was_in_short <- FALSE; was_in_long <- FALSE
        } else {
            if (short) {
                pnl_pct <- shares_traded * (enter_price - row$close) / notional
                if (!is.na(pnl_pct) && pnl_pct < -0.05) {
                    df$PnL[i] <- shares_traded*(enter_price-row$close)
                    df$Trade[i] <- "SExitR"
                    short <- FALSE; shares_traded <- 0
                    was_in_short <- TRUE; was_in_long <- FALSE
                } else {
                    df$Trade[i] <- "In-S"
                    if (!is.na(row$dc_pband) && !is.na(last_dc_pband) &&
                        (row$dc_pband <= last_dc_pband || row$dc_wband >= last_dc_wband ||
                         row$close - row$dc_lband < row$dc_mband - row$close))
                        exit_signal <- TRUE
                    if (exit_signal && !is.na(row$dc_pband) && row$dc_pband > last_dc_pband &&
                        row$dc_wband < last_dc_wband) {
                        df$PnL[i] <- shares_traded*(enter_price-row$close)
                        df$Trade[i] <- "SExit"
                        short <- FALSE; shares_traded <- 0; exit_signal <- FALSE
                        was_in_short <- TRUE; was_in_long <- FALSE
                    }
                }
            } else if (long_pos) {
                pnl_pct <- shares_traded * (row$close - enter_price) / notional
                if (!is.na(pnl_pct) && pnl_pct < -0.05) {
                    df$PnL[i] <- shares_traded*(row$close-enter_price)
                    df$Trade[i] <- "LExitR"
                    long_pos <- FALSE; shares_traded <- 0
                    was_in_long <- TRUE; was_in_short <- FALSE
                } else {
                    df$Trade[i] <- "In-L"
                    if (!is.na(row$dc_pband) && !is.na(last_dc_pband) &&
                        (row$dc_pband >= last_dc_pband || row$dc_wband >= last_dc_wband ||
                         row$dc_hband - row$close < row$close - row$dc_mband))
                        exit_signal <- TRUE
                    if (exit_signal && !is.na(row$dc_pband) && row$dc_pband < last_dc_pband &&
                        row$dc_wband < last_dc_wband) {
                        df$PnL[i] <- shares_traded*(row$close-enter_price)
                        df$Trade[i] <- "LExit"
                        long_pos <- FALSE; shares_traded <- 0; exit_signal <- FALSE
                        was_in_long <- TRUE; was_in_short <- FALSE
                    }
                }
            }
        }
        last_dc_pband <- df$dc_pband[i]
        last_dc_wband <- df$dc_wband[i]
    }

    df$CumuPnL    <- cumsum(df$PnL)
    df$CumuPnL[1] <- 10000
    df
}